# 31. SFT — 지도 미세조정

> **제31장** · **이론편 대응: 22장 (Fine-Tuning)**
> **예상 소요**: 80분 (학습 3~5분)
> **필요 사양**: **[CPU]** 로 실행 가능 (GPU 있으면 더 빠름)
> **추가 설치**: 없음 (23장의 transformers 사용)
> **다운로드**: DistilGPT-2 약 330MB (자동)

---

## 이 장에서 하는 일

28장의 RAG가 "지식을 공급하는" 방법이라면, 파인튜닝은 **모델 자체를 바꾸는** 방법이다.

| 절 | 하는 일 | 이론편 대응 |
|---|---|---|
| 1 | 사전학습 모델의 문제 확인 | 22.1절 |
| 2 | **SFT 데이터 형식** | 22.2절 |
| 3 | **손실 마스킹 직접 구현** ★ | 22.2절 |
| 4 | 학습 실행 | 22.2절 |
| 5 | 학습 전후 비교 | 22.2절 |
| 6 | 메모리 사용량 측정 | 22.5절 |
| 7 | 과대적합 확인 | 8.5절 |
| 8 | SFT의 한계 | 22.6절 |

**3절이 핵심이다.** 이론편 22.2절에서 "지시 부분은 손실에서 제외한다"고 했는데,
그것을 코드로 어떻게 구현하는지 확인한다.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform
import time

_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} / 장치: {device}")

from transformers import AutoTokenizer, AutoModelForCausalLM

# CPU에서도 돌아가도록 작은 모델을 쓴다
MODEL_NAME = "distilgpt2"

print(f"\n모델 불러오는 중: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token      # GPT-2 계열은 pad 토큰이 없다

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
n_params = sum(p.numel() for p in model.parameters())
print(f"파라미터: {n_params/1e6:.0f}M")
print()
print("23장의 GPT-2(124M)보다 작다 — CPU 학습을 위해 고른 것이다.")
print("실무에서는 훨씬 큰 모델을 쓰지만, 원리는 같다.")

---

## 1. 사전학습 모델의 문제 — 이론편 22.1절

23장에서 GPT-2로 문장을 생성했다. 그런데 **질문을 하면 답을 하지 않는다.**

이론편 22.1절에서 다룬 대로, 사전학습 모델은 "다음 단어 예측"만 배웠기 때문이다.
인터넷 텍스트에는 질문 뒤에 또 다른 질문이 오는 경우도 많다.

직접 확인해 보자.

In [ ]:
import torch


def generate(model, prompt, max_new_tokens=40, temperature=0.0):
    """텍스트 생성 (24장 방식)"""
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0),
            temperature=temperature if temperature > 0 else None,
            pad_token_id=tokenizer.eos_token_id,
        )
    full = tokenizer.decode(out[0], skip_special_tokens=True)
    return full[len(prompt):].strip()


model = model.to(device)

print("=" * 70)
print("사전학습 모델에게 질문하면 (이론편 22.1절)")
print("=" * 70)

questions = [
    "What is Python?",
    "How do I learn programming?",
]

for q in questions:
    print(f"\n질문: {q}")
    answer = generate(model, q, max_new_tokens=35)
    print(f"응답: {answer}")

print()
print("-" * 70)
print("답변이라기보다 '이어 쓰기'에 가깝다.")
print("  질문 뒤에 또 다른 질문이 오거나, 문맥 없는 문장이 이어진다.")
print()
print("원인: 이 모델은 '질문에 답하라'고 배운 적이 없다.")
print("  인터넷 텍스트를 그대로 이어 쓰도록만 학습되었다.")

### 해법 — 지시-응답 쌍으로 가르친다

이론편 22.2절의 SFT(Supervised Fine-Tuning)가 이 문제를 푼다.

**"이런 질문에는 이렇게 답하라"는 예시를 잔뜩 보여주는 것**이다.

$$L = -\sum_{t}\log P(y_t \mid x,\ y_{<t})$$

여기서 $x$는 지시문, $y$는 모범 응답이다. **$y$의 토큰들만 합하는 것**이 핵심이며,
3절에서 이것을 코드로 구현한다.

---

## 2. SFT 데이터 형식 — 이론편 22.2절

학습 데이터는 **지시와 응답의 쌍**이다. 24장 5절에서 다룬 대화 형식으로 감싼다.

이 장에서는 구조를 명확히 보기 위해 단순한 형식을 쓴다.

```
Q: 질문 내용
A: 응답 내용
```

실제 모델들은 `<|im_start|>` 같은 특수 토큰을 쓰지만(24장 5절), **원리는 같다** —
"어디까지가 질문이고 어디부터가 답인가"를 표시하는 것이다.

In [ ]:
# 학습 데이터 — 프로그래밍 관련 질의응답
train_data = [
    ("What is Python?",
     "Python is a high-level programming language known for its simple syntax."),
    ("What is a variable?",
     "A variable is a named container that stores a value in memory."),
    ("What is a function?",
     "A function is a reusable block of code that performs a specific task."),
    ("What is a loop?",
     "A loop is a control structure that repeats a block of code."),
    ("What is a list?",
     "A list is an ordered collection of items that can be changed."),
    ("What is machine learning?",
     "Machine learning is a method where computers learn patterns from data."),
    ("What is a neural network?",
     "A neural network is a model composed of connected layers of nodes."),
    ("What is debugging?",
     "Debugging is the process of finding and fixing errors in code."),
]

# 평가용 (학습에 쓰지 않음)
eval_data = [
    ("What is a dictionary?",
     "A dictionary is a collection of key-value pairs."),
    ("What is an algorithm?",
     "An algorithm is a step-by-step procedure to solve a problem."),
]

PROMPT_TEMPLATE = "Q: {question}\nA:"


def format_example(question, answer):
    """지시-응답 쌍을 하나의 문자열로"""
    prompt = PROMPT_TEMPLATE.format(question=question)
    completion = f" {answer}{tokenizer.eos_token}"
    return prompt, completion


print("=" * 70)
print("SFT 데이터 형식 (이론편 22.2절)")
print("=" * 70)
print(f"학습 데이터: {len(train_data)}쌍")
print(f"평가 데이터: {len(eval_data)}쌍")
print()

q, a = train_data[0]
prompt, completion = format_example(q, a)

print("한 건의 예")
print(f"  프롬프트 부분: {repr(prompt)}")
print(f"  응답 부분    : {repr(completion)}")
print()
print("합쳐서 모델에 넣을 문자열")
print(f"  {repr(prompt + completion)}")
print()
print("EOS 토큰을 뒤에 붙이는 이유")
print("  '여기서 답이 끝난다'를 모델이 배워야 한다.")
print("  없으면 답을 마치지 못하고 계속 이어 쓴다.")

---

## 3. 손실 마스킹 직접 구현 ★ — 이론편 22.2절

**이 장에서 가장 중요한 부분이다.**

이론편 22.2절에서 이렇게 설명했다.

> 위 데이터 전체를 다음 토큰 예측으로 학습시키면, 모델은 응답뿐 아니라
> **질문까지 잘 만들어내도록** 배우게 된다. 우리가 원하는 것은 그것이 아니다.

그래서 **응답 부분에서만 손실을 계산**한다. 코드로는 어떻게 할까.

In [ ]:
import torch


def prepare_example(question, answer, max_length=64, mask_prompt=True):
    """SFT 학습용 데이터 하나를 만든다 (이론편 22.2절)

    mask_prompt=True 면 프롬프트 부분의 라벨을 -100 으로 채운다.
    -100 은 PyTorch CrossEntropyLoss 의 기본 ignore_index 값으로,
    "이 위치는 손실 계산에서 제외하라"는 뜻이다.
    """
    prompt, completion = format_example(question, answer)

    prompt_ids = tokenizer.encode(prompt)
    completion_ids = tokenizer.encode(completion)

    input_ids = prompt_ids + completion_ids

    if mask_prompt:
        # 프롬프트 자리는 -100, 응답 자리는 실제 토큰 ID
        labels = [-100] * len(prompt_ids) + completion_ids
    else:
        labels = input_ids.copy()

    # 길이 제한
    input_ids = input_ids[:max_length]
    labels = labels[:max_length]

    return {
        "input_ids": input_ids,
        "labels": labels,
        "n_prompt": len(prompt_ids),
        "n_completion": len(completion_ids),
    }


ex = prepare_example(*train_data[0])

print("=" * 70)
print("손실 마스킹 구조")
print("=" * 70)
print(f"프롬프트 토큰: {ex['n_prompt']}개")
print(f"응답 토큰    : {ex['n_completion']}개")
print(f"전체         : {len(ex['input_ids'])}개")
print()

print(f"{'위치':<6}{'토큰':<22}{'input_ids':<14}{'labels':<12}{'손실 계산'}")
print("-" * 70)
for i in range(len(ex["input_ids"])):
    tok_str = repr(tokenizer.decode([ex["input_ids"][i]]))
    label = ex["labels"][i]
    calc = "제외" if label == -100 else "포함"
    marker = "  ← 여기부터 학습" if i == ex["n_prompt"] else ""
    print(f"{i:<6}{tok_str:<22}{ex['input_ids'][i]:<14}{label:<12}{calc}{marker}")
print("-" * 70)
print()
print("-100 이 채워진 자리는 손실에 반영되지 않는다.")
print("모델은 '질문 다음에 무엇이 오는가'만 배우고,")
print("'질문 자체를 어떻게 만드는가'는 배우지 않는다.")

In [ ]:
import torch
import torch.nn as nn

print("=" * 65)
print("-100 이 왜 무시되는가")
print("=" * 65)

print(f"nn.CrossEntropyLoss 의 기본 ignore_index: {nn.CrossEntropyLoss().ignore_index}")
print()
print("PyTorch가 정한 약속이다. 라벨이 -100인 위치는 손실 계산에서 빠진다.")
print()

# 직접 확인
logits = torch.randn(1, 5, 100)      # (배치, 토큰 5개, 어휘 100)
labels_all = torch.tensor([[10, 20, 30, 40, 50]])
labels_masked = torch.tensor([[-100, -100, 30, 40, 50]])

crit = nn.CrossEntropyLoss()
loss_all = crit(logits.view(-1, 100), labels_all.view(-1))
loss_masked = crit(logits.view(-1, 100), labels_masked.view(-1))

print(f"전체 5개 토큰으로 계산: {loss_all.item():.4f}")
print(f"뒤 3개만으로 계산     : {loss_masked.item():.4f}")
print()

# 뒤 3개만 직접 계산해 대조
loss_manual = crit(logits[:, 2:].reshape(-1, 100), labels_all[:, 2:].reshape(-1))
print(f"뒤 3개를 직접 잘라 계산: {loss_manual.item():.4f}")
print(f"마스킹 결과와 같은가: {abs(loss_masked.item() - loss_manual.item()) < 1e-5}")
print()
print("[OK] -100 위치가 실제로 제외된다")

In [ ]:
import torch

print("=" * 70)
print("마스킹 유무에 따른 손실 차이 — 실제 모델로")
print("=" * 70)

q, a = train_data[0]
ex_masked = prepare_example(q, a, mask_prompt=True)
ex_full = prepare_example(q, a, mask_prompt=False)

model.eval()
with torch.no_grad():
    ids = torch.tensor([ex_masked["input_ids"]]).to(device)

    loss_m = model(input_ids=ids,
                   labels=torch.tensor([ex_masked["labels"]]).to(device)).loss
    loss_f = model(input_ids=ids,
                   labels=torch.tensor([ex_full["labels"]]).to(device)).loss

print(f"질문: {q}")
print()
print(f"{'방식':<24}{'손실':<12}{'계산 대상'}")
print("-" * 70)
print(f"{'마스킹 (응답만)':<24}{loss_m.item():<12.4f}{ex_masked['n_completion']}개 토큰")
print(f"{'마스킹 없음 (전체)':<24}{loss_f.item():<12.4f}{len(ex_full['input_ids'])}개 토큰")
print("-" * 70)
print()
print("값이 다르다 — 계산에 포함되는 토큰이 다르기 때문이다.")
print()
print("마스킹 없이 학습하면 어떻게 되나")
print("  모델이 'Q: What is Python?' 같은 문장을 만드는 것도 함께 배운다.")
print("  질문을 생성하는 능력은 우리가 원하는 것이 아니다.")
print()
print("이론편 22.2절의 비유: 시험공부에서 외워야 할 것은 답안이지 문제가 아니다.")

---

## 4. 학습 실행 — 이론편 22.2절

준비가 끝났으니 실제로 학습시킨다. 12장에서 익힌 표준 학습 루프를 쓴다.

**배치 처리를 위해 패딩이 필요하다.** 문장 길이가 제각각이므로,
짧은 것을 채워 길이를 맞춘다. **패딩 자리도 손실에서 제외**해야 한다.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader


class SFTDataset(Dataset):
    # SFT 학습용 데이터셋

    def __init__(self, data, max_length=64, mask_prompt=True):
        self.examples = [
            prepare_example(q, a, max_length, mask_prompt) for q, a in data
        ]

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


def collate_fn(batch):
    # 배치로 묶으며 길이를 맞춘다 (패딩)
    #
    # 두 가지를 채운다:
    #   input_ids → pad 토큰 (모델이 읽기는 함)
    #   labels    → -100 (손실에서 제외)
    max_len = max(len(b["input_ids"]) for b in batch)
    pad_id = tokenizer.pad_token_id

    input_ids, labels, attention_mask = [], [], []
    for b in batch:
        n_pad = max_len - len(b["input_ids"])
        input_ids.append(b["input_ids"] + [pad_id] * n_pad)
        labels.append(b["labels"] + [-100] * n_pad)          # 패딩도 제외
        attention_mask.append([1] * len(b["input_ids"]) + [0] * n_pad)

    return {
        "input_ids": torch.tensor(input_ids),
        "labels": torch.tensor(labels),
        "attention_mask": torch.tensor(attention_mask),
    }


train_ds = SFTDataset(train_data)
train_dl = DataLoader(train_ds, batch_size=2, shuffle=True, collate_fn=collate_fn)

print("=" * 70)
print("데이터로더 확인")
print("=" * 70)
batch = next(iter(train_dl))
print(f"배치 모양")
for k, v in batch.items():
    print(f"  {k:<16}{tuple(v.shape)}")
print()

print("첫 샘플의 labels (앞 12개)")
print(f"  {batch['labels'][0][:12].tolist()}")
print()
print("attention_mask 의 역할")
print("  1인 자리만 Attention 계산에 쓴다 (22장 1절의 마스킹과 같은 원리).")
print("  패딩 자리를 모델이 실제 토큰으로 착각하지 않게 한다.")

In [ ]:
import torch
import time

# 학습 전 상태를 기록해 둔다 (5절에서 비교)
before_answers = {}
model.eval()
for q, _ in eval_data + [train_data[0]]:
    prompt = PROMPT_TEMPLATE.format(question=q)
    before_answers[q] = generate(model, prompt, max_new_tokens=30)

print("=" * 70)
print("학습 시작")
print("=" * 70)

EPOCHS = 8
LR = 5e-5

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
model.train()

history = []
t0 = time.time()

for epoch in range(EPOCHS):
    total_loss, n_batch = 0.0, 0
    for batch in train_dl:
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()
        outputs = model(**batch)          # labels 를 넣으면 손실이 함께 나온다
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        n_batch += 1

    avg = total_loss / n_batch
    history.append(avg)
    print(f"  에폭 {epoch+1:2}/{EPOCHS}: 손실 {avg:.4f}  ({time.time()-t0:.0f}초)")

print("-" * 70)
print(f"소요 시간: {time.time()-t0:.0f}초")
print(f"초기 손실 {history[0]:.4f} → 최종 손실 {history[-1]:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, len(history)+1), history, marker="o", linewidth=2, color="#1E40AF")
ax.set_xlabel("에폭")
ax.set_ylabel("손실")
ax.set_title("SFT 학습 곡선")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("손실이 빠르게 떨어진다.")
print()
print("주의: 학습 데이터가 8개뿐이라 금방 외워 버린다.")
print("  손실이 낮다고 좋은 것이 아니다 — 7절에서 과대적합을 확인한다.")

---

## 5. 학습 전후 비교 — 이론편 22.2절

**같은 질문에 대한 답이 어떻게 달라졌는지** 확인한다.

In [ ]:
import torch

print("=" * 78)
print("학습 전후 비교")
print("=" * 78)

model.eval()

# 학습에 쓴 질문
q_seen = train_data[0][0]
prompt = PROMPT_TEMPLATE.format(question=q_seen)
after = generate(model, prompt, max_new_tokens=30)

print(f"\n[학습에 사용한 질문] {q_seen}")
print(f"  학습 전: {before_answers[q_seen][:70]}")
print(f"  학습 후: {after[:70]}")
print(f"  정답   : {train_data[0][1][:70]}")

# 학습에 없던 질문
print()
print("=" * 78)
print("[학습에 없던 질문]")
for q, a in eval_data:
    prompt = PROMPT_TEMPLATE.format(question=q)
    after = generate(model, prompt, max_new_tokens=30)
    print(f"\n질문: {q}")
    print(f"  학습 전: {before_answers[q][:70]}")
    print(f"  학습 후: {after[:70]}")
    print(f"  참고 답: {a[:70]}")

print()
print("=" * 78)
print("무엇이 달라졌나")
print("  1) 'Q: ... A:' 형식을 인식하고 답변 위치에 답을 쓴다")
print("  2) 문장을 적절한 곳에서 끝낸다 (EOS 학습 효과)")
print()
print("무엇이 달라지지 않았나")
print("  모르는 내용을 새로 알게 된 것은 아니다.")
print("  → 8절에서 다룬다.")

---

## 6. 메모리 사용량 — 이론편 22.5절

이론편 22.5절에서 파인튜닝에 필요한 메모리를 계산했다.

$$M_{\text{학습}} = \underbrace{M_{\text{가중치}}}_{\times 1} + \underbrace{M_{\text{그래디언트}}}_{\times 1} + \underbrace{M_{\text{옵티마이저}}}_{\times 2}$$

**실제로 측정해 확인한다.**

In [ ]:
import torch

print("=" * 70)
print("메모리 사용량 (이론편 22.5절)")
print("=" * 70)

n_params = sum(p.numel() for p in model.parameters())
bytes_per = next(model.parameters()).element_size()

weights_mb = n_params * bytes_per / 1024**2
grads_mb = weights_mb                      # 파라미터마다 그래디언트 하나
adam_mb = weights_mb * 2                   # Adam은 상태 두 개 (이론편 11.1절)

print(f"모델      : {MODEL_NAME}")
print(f"파라미터  : {n_params/1e6:.1f}M")
print(f"자료형    : {next(model.parameters()).dtype} ({bytes_per}바이트)")
print()
print(f"{'항목':<24}{'크기(MB)':<14}{'배수'}")
print("-" * 70)
print(f"{'가중치':<24}{weights_mb:<14.1f}x1")
print(f"{'그래디언트':<24}{grads_mb:<14.1f}x1")
print(f"{'Adam 상태 (m, v)':<24}{adam_mb:<14.1f}x2")
print("-" * 70)
print(f"{'합계':<24}{weights_mb+grads_mb+adam_mb:<14.1f}x4")
print()

# 실제 옵티마이저 상태 확인
opt_state_size = 0
for group in optimizer.param_groups:
    for p in group["params"]:
        state = optimizer.state.get(p, {})
        for k, v in state.items():
            if torch.is_tensor(v):
                opt_state_size += v.numel() * v.element_size()

print(f"실제 옵티마이저 상태: {opt_state_size/1024**2:.1f} MB")
print(f"예상값과 비교      : {adam_mb:.1f} MB")
print()
print("여기에 활성값(순전파 중간 결과)이 더해진다.")
print("  11장에서 봤듯 역전파에 필요해 저장해 두기 때문이다.")

In [ ]:
print("=" * 70)
print("더 큰 모델이라면 (이론편 22.5절 표)")
print("=" * 70)
print(f"{'모델':<12}{'가중치(FP16)':<16}{'전체 파인튜닝':<18}{'8GB GPU'}")
print("-" * 70)

for n, name in [(0.082e9, "DistilGPT2"), (0.124e9, "GPT-2"),
                (1.5e9, "1.5B"), (7e9, "7B"), (13e9, "13B")]:
    w = n * 2 / 1024**3          # FP16
    total = w * 4                # 가중치 + 그래디언트 + Adam 2개
    verdict = "가능" if total < 6.5 else "불가능"
    print(f"{name:<12}{w:<16.2f}{total:<18.1f}{verdict}")

print("-" * 70)
print()
print("7B 모델의 전체 파인튜닝에는 약 52GB가 필요하다.")
print("일반적인 GPU로는 불가능하다.")
print()
print("→ 이것이 이론편 22.3~22.5절에서 LoRA·QLoRA가 나온 이유다.")
print("  32장에서 다룬다.")

---

## 7. 과대적합 확인 — 이론편 8.5절

학습 손실이 0에 가까워졌다. **좋은 신호일까?**

데이터가 8개뿐이므로 **외워버렸을 가능성**이 크다. 이론편 8.5절에서 다룬 과대적합이다.

In [ ]:
import torch
import numpy as np

print("=" * 70)
print("학습 데이터 vs 평가 데이터의 손실")
print("=" * 70)


def compute_loss(data):
    ds = SFTDataset(data)
    dl = DataLoader(ds, batch_size=2, shuffle=False, collate_fn=collate_fn)
    model.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for batch in dl:
            batch = {k: v.to(device) for k, v in batch.items()}
            total += model(**batch).loss.item()
            n += 1
    return total / n


train_loss = compute_loss(train_data)
eval_loss = compute_loss(eval_data)

print(f"{'데이터':<20}{'손실':<14}{'설명'}")
print("-" * 70)
print(f"{'학습 데이터':<20}{train_loss:<14.4f}학습에 사용")
print(f"{'평가 데이터':<20}{eval_loss:<14.4f}학습에 미사용")
print("-" * 70)
print(f"격차: {eval_loss - train_loss:.4f}")
print()

if eval_loss > train_loss * 2:
    print("[과대적합] 평가 손실이 학습 손실보다 훨씬 크다.")
    print("  모델이 학습 데이터를 외웠을 뿐, 일반화하지 못하고 있다.")
else:
    print("격차가 크지 않다.")

print()
print("원인: 학습 데이터가 8개뿐이다.")
print()
print("실무에서 필요한 데이터 양")
print("  최소 수백 건, 보통 수천~수만 건")
print("  품질이 양보다 중요하다는 연구 결과도 많다 (이론편 22.2절)")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 에폭별로 두 손실을 추적하는 실험 (다시 학습)
print("=" * 70)
print("에폭에 따른 과대적합 진행")
print("=" * 70)

from transformers import AutoModelForCausalLM

torch.manual_seed(42)
model2 = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
opt2 = torch.optim.AdamW(model2.parameters(), lr=5e-5)

train_dl2 = DataLoader(SFTDataset(train_data), batch_size=2,
                       shuffle=True, collate_fn=collate_fn)

def eval_loss_of(m, data):
    dl = DataLoader(SFTDataset(data), batch_size=2, shuffle=False, collate_fn=collate_fn)
    m.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for b in dl:
            b = {k: v.to(device) for k, v in b.items()}
            total += m(**b).loss.item(); n += 1
    return total / n

tr_hist, ev_hist = [], []
for epoch in range(8):
    model2.train()
    for batch in train_dl2:
        batch = {k: v.to(device) for k, v in batch.items()}
        opt2.zero_grad()
        model2(**batch).loss.backward()
        opt2.step()
    tr_hist.append(eval_loss_of(model2, train_data))
    ev_hist.append(eval_loss_of(model2, eval_data))
    print(f"  에폭 {epoch+1}: 학습 {tr_hist[-1]:.4f}  평가 {ev_hist[-1]:.4f}")

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(range(1, 9), tr_hist, marker="o", label="학습 손실", linewidth=2)
ax.plot(range(1, 9), ev_hist, marker="s", label="평가 손실",
        linewidth=2, linestyle="--")
ax.set_xlabel("에폭")
ax.set_ylabel("손실")
ax.set_title("과대적합 진행 (이론편 8.5절, 11.6절)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

best = int(np.argmin(ev_hist)) + 1
print()
print(f"평가 손실이 가장 낮은 에폭: {best}")
print("  이 지점에서 멈추는 것이 조기 종료(early stopping)다 (이론편 11.6절).")
print()
print("03장 3절의 손실 곡선 패턴과 같다.")

---

## 8. SFT의 한계 — 이론편 22.6절

SFT로 무엇이 되고 무엇이 안 되는지 정리한다.

In [ ]:
print("=" * 70)
print("SFT로 되는 것과 안 되는 것 (이론편 22.6절)")
print("=" * 70)
print()
print(f"{'항목':<26}{'SFT':<10}{'설명'}")
print("-" * 70)
rows = [
    ("답변 형식·말투",        "가능",   "예시로 보여주면 따라 한다"),
    ("특정 작업 수행 방식",    "가능",   "분류·요약 등 패턴 학습"),
    ("지시를 따르는 능력",     "가능",   "이 장에서 확인"),
    ("새로운 지식 주입",       "제한적", "많은 데이터·반복 필요"),
    ("최신 정보 반영",        "어려움", "바뀔 때마다 재학습"),
    ("근거 제시",            "불가",   "출처를 알 수 없다"),
]
for a, b, c in rows:
    print(f"{a:<26}{b:<10}{c}")
print("-" * 70)
print()
print("[핵심] SFT는 **방식**을 가르치는 데 강하고, **지식**을 넣는 데는 약하다.")
print()
print("지식이 필요하면 28장의 RAG가 낫다:")
print("  - 문서만 갱신하면 최신 정보 반영")
print("  - 출처를 보여줄 수 있다")
print("  - 재학습이 필요 없다")

In [ ]:
print("=" * 70)
print("RAG와 SFT를 함께 쓰기 (이론편 22.6절)")
print("=" * 70)
print()
print("둘은 배타적이지 않다. 실무에서는 함께 쓰는 경우가 많다.")
print()
print("  SFT로 → 우리 회사 말투, 답변 형식, 출처 표기 방식을 익힌다")
print("  RAG로 → 최신 문서를 공급한다")
print()
print("-" * 70)
print("선택 기준")
print()
print(f"{'상황':<34}{'권장'}")
print("-" * 70)
choices = [
    ("자주 바뀌는 정보를 다뤄야 함",        "RAG"),
    ("답변에 근거를 표시해야 함",           "RAG"),
    ("특정 형식·말투를 일관되게 유지",      "SFT"),
    ("전문 분야 용어를 이해해야 함",        "SFT"),
    ("응답 속도가 중요 (검색 단계 부담)",    "SFT"),
    ("둘 다 필요",                        "SFT + RAG"),
]
for a, b in choices:
    print(f"{a:<34}{b}")
print("-" * 70)
print()
print("다음 장에서는 SFT를 더 적은 자원으로 하는 방법을 다룬다.")
print("6절에서 봤듯 전체 파인튜닝은 메모리 부담이 크기 때문이다.")

---

## 9. 정리

### 확인한 이론편 내용

| 이론편 절 | 내용 | 결과 |
|---|---|---|
| 22.1 | 사전학습 모델은 질문에 답하지 않음 | 확인 ✓ |
| **22.2** | **손실 마스킹 — 응답만 학습** | **구현·검증** ✓ |
| 22.2 | SFT로 지시를 따르게 됨 | 전후 비교 ✓ |
| 22.5 | 학습 메모리 = 가중치 x4 | 측정 ✓ |
| 8.5 | 데이터가 적으면 과대적합 | 곡선 확인 ✓ |
| 22.6 | SFT는 지식보다 방식 | 정리 ✓ |

### 손실 마스킹 요약

```python
prompt_ids = tokenizer.encode(prompt)
completion_ids = tokenizer.encode(completion)

input_ids = prompt_ids + completion_ids
labels    = [-100] * len(prompt_ids) + completion_ids
#            ^^^^^^ 프롬프트는 손실에서 제외
```

**패딩 자리도 -100으로** 채워야 한다는 점을 잊지 말자.

### 기억할 것

| 항목 | 요점 |
|---|---|
| `-100` | PyTorch의 `ignore_index` 기본값 |
| EOS 토큰 | 답을 마치는 법을 배우게 함 |
| `attention_mask` | 패딩 자리를 Attention에서 제외 |
| 학습 메모리 | 가중치의 4배 (+활성값) |
| 데이터 양 | 최소 수백 건, 품질이 중요 |
| SFT의 강점 | 지식보다 **방식** |

### 다음 장

**32. LoRA와 QLoRA — 적은 자원으로 파인튜닝** — 이론편 22.3~22.5절.
6절에서 본 메모리 문제를 푸는 방법이다.
**이론편 22.4절에서 계산한 파라미터 비율 0.39%**를 직접 확인한다.